In [1]:
import torch

assert torch.cuda.is_available(), "GPU not enabled — go to Settings → Accelerator"
print(f"✓ CUDA available: {torch.version.cuda}")
print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✓ CUDA available: 12.8
✓ GPU: Tesla T4
✓ VRAM: 15.6 GB


In [2]:
import subprocess, sys

def pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip("PyYAML")
pip("librosa")
pip("soundfile")
pip("einops")
pip("importlib-resources")
pip("torchcontrib")

print("✓ Dependencies installed")

✓ Dependencies installed


In [3]:
import os, json
from pathlib import Path

# ── Install Kaggle API ────────────────────────────────────────────────────────
!pip install -q kaggle

# ── Download & extract ────────────────────────────────────────────────────────
KAGGLE_DATASET  = 'awsaf49/asvpoof-2019-dataset'
DOWNLOAD_DIR    = Path('/input/asvspoof2019_download')
EXTRACT_DIR     = Path('/input/ASVspoof2019')

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f'Downloading {KAGGLE_DATASET} ...')
!kaggle datasets download -d {KAGGLE_DATASET} -p {DOWNLOAD_DIR} --unzip

# Locate the LA folder (handle varying zip layouts)
# The Kaggle dataset unpacks as: LA/ at root, or nested inside a folder.
la_candidates = sorted(DOWNLOAD_DIR.rglob('ASVspoof2019_LA_train'))
if la_candidates:
    DATA_ROOT = str(la_candidates[0].parent)
else:
    # Fallback: assume flat layout
    DATA_ROOT = str(DOWNLOAD_DIR)

print(f'DATA_ROOT resolved to: {DATA_ROOT}')
print('Contents:', os.listdir(DATA_ROOT))

Dataset URL: https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset
License(s): ODC Attribution License (ODC-By)
100%|███████████████████████████████████████| 23.6G/23.6G [02:00<00:00, 209MB/s]

DATA_ROOT resolved to: /input/asvspoof2019_download/LA/LA
Contents: ['ASVspoof2019_LA_train', 'ASVspoof2019_LA_asv_protocols', 'ASVspoof2019_LA_asv_scores', 'README.LA.txt', 'ASVspoof2019_LA_eval', 'ASVspoof2019_LA_cm_protocols', 'ASVspoof2019_LA_dev']


In [4]:
PROJECT_DIR  = Path("/kaggle/working/project4")
DATA_DIR     = Path("/input/asvspoof2019_download/LA/LA")  # adjust to your dataset slug
MODEL_DIR    = PROJECT_DIR / "rawnet2_models"
AASIST_DIR   = PROJECT_DIR / "aasist"
LOG_FILE     = PROJECT_DIR / "rawnet2_training.log"
EVAL_OUTPUT  = PROJECT_DIR / "rawnet2_eval_output.txt"
CONFIG_PATH  = AASIST_DIR / "config" / "RawNet2.conf"

for d in [MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Paths set")
print(f"  Project : {PROJECT_DIR}")
print(f"  Data    : {DATA_DIR}")
print(f"  Models  : {MODEL_DIR}")

# Sanity-check dataset
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR} — check your Kaggle dataset attachment"
print("✓ Dataset directory found")

✓ Paths set
  Project : /kaggle/working/project4
  Data    : /input/asvspoof2019_download/LA/LA
  Models  : /kaggle/working/project4/rawnet2_models
✓ Dataset directory found


In [5]:
import subprocess

if not AASIST_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/clovaai/aasist.git", str(AASIST_DIR)],
        check=True
    )
    print("✓ AASIST repo cloned")
else:
    print("✓ AASIST repo already present — skipping clone")

print(f"  Contents: {[f.name for f in AASIST_DIR.iterdir()]}")

Cloning into '/kaggle/working/project4/aasist'...


✓ AASIST repo cloned
  Contents: ['requirements.txt', 'download_dataset.py', 'README.md', 'models', 'evaluation.py', '.gitignore', 'data_utils.py', '.git', 'config', 'main.py', 'LICENSE', 'NOTICE', 'utils.py']


In [6]:
# Grep main.py for every config[...] access so we know the complete schema.
# Run this BEFORE writing the config — output tells you exactly what keys are needed.
import re
from pathlib import Path

main_src = (AASIST_DIR / "main.py").read_text()

# Find all config["key"] and config['key'] accesses
keys = re.findall(r'config\[["\'](.*?)["\']\]', main_src)
top_level = sorted(set(keys))

# Also find optim_config and model_config sub-keys from utils.py
utils_src = (AASIST_DIR / "utils.py").read_text()
optim_keys = re.findall(r'optim_config\[["\'](.*?)["\']\]', utils_src)
model_keys  = re.findall(r'd_args\[["\'](.*?)["\']\]', utils_src)

print("=== Top-level config keys ===")
for k in top_level: print(f"  {k}")

print("\n=== optim_config sub-keys ===")
for k in sorted(set(optim_keys)): print(f"  {k}")

print("\n=== model d_args sub-keys (from utils.py) ===")
for k in sorted(set(model_keys)): print(f"  {k}")

=== Top-level config keys ===
  architecture
  asv_score_path
  batch_size
  database_path
  epochs
  eval_all_best
  eval_output
  freq_aug
  model_config
  model_path
  num_epochs
  optim_config
  steps_per_epoch
  track

=== optim_config sub-keys ===
  T0
  Tmult
  amsgrad
  base_lr
  betas
  epochs
  lr_decay
  lr_min
  milestones
  momentum
  nesterov
  optimizer
  scheduler
  steps_per_epoch
  weight_decay

=== model d_args sub-keys (from utils.py) ===


In [7]:
import json
from pathlib import Path

config_dir = AASIST_DIR / "config"

# Print both RawNet2 configs that shipped with the repo
for name in ["RawNet2.conf", "RawNet2_baseline.conf"]:
    p = config_dir / name
    if p.exists():
        print(f"\n{'='*50}")
        print(f" {name}")
        print(f"{'='*50}")
        print(p.read_text())

# Also confirm the model class name
print("\n=== Class names in RawNet2Spoof.py ===")
import re
src = (AASIST_DIR / "models" / "RawNet2Spoof.py").read_text()
classes = re.findall(r'^class (\w+)', src, re.MULTILINE)
print(f"  {classes}")


 RawNet2_baseline.conf
{
    "database_path": "./LA/",
    "asv_score_path": "ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt",
    "model_path": "/home1/irteam/jeeweon/git/AsvSpoofDetection/exp_result/LAmodelRawNet2Spoof_ep100_bs32_lr0.0001/weights/best.pth",
    "batch_size": 32,
    "lr": 0.0001,
    "weight_decay": 0.0001,
    "num_epochs": 100,
    "loss": "CCE",
    "track": "LA",
    "eval_output": "eval_scores_using_best_dev_model.txt",
    "cudnn_deterministic_toggle": "True",
    "cudnn_benchmark_toggle": "False",
    "model_config": {
        "architecture": "RawNet2Spoof",
        "nb_samp": 64600,
        "first_conv": 1024,
        "in_channels": 1,
        "filts": [20, [20, 20], [20, 128], [128, 128]],
        "blocks": [2, 4],
        "nb_fc_node": 1024,
        "gru_node": 1024,
        "nb_gru_layer": 3,
        "nb_classes": 2
    },
    "optim_config": {
        "optimizer": "adam", 
        "amsgrad": "False",
        "base_lr": 0.0001,
    

In [8]:
import os
from pathlib import Path

print("=== aasist/ top-level ===")
for f in sorted(AASIST_DIR.iterdir()):
    print(f"  {f.name}{'/' if f.is_dir() else ''}")

print("\n=== aasist/models/ (if it exists) ===")
models_dir = AASIST_DIR / "models"
if models_dir.exists():
    for f in sorted(models_dir.iterdir()):
        print(f"  {f.name}")
else:
    print("  models/ directory does NOT exist")

print("\n=== aasist/config/ (if it exists) ===")
config_dir = AASIST_DIR / "config"
if config_dir.exists():
    for f in sorted(config_dir.iterdir()):
        print(f"  {f.name}")
else:
    print("  config/ directory does NOT exist")

print("\n=== .gitignore contents ===")
gi = AASIST_DIR / ".gitignore"
if gi.exists():
    print(gi.read_text())

=== aasist/ top-level ===
  .git/
  .gitignore
  LICENSE
  NOTICE
  README.md
  config/
  data_utils.py
  download_dataset.py
  evaluation.py
  main.py
  models/
  requirements.txt
  utils.py

=== aasist/models/ (if it exists) ===
  AASIST.py
  RawNet2Spoof.py
  RawNetGatSpoofST.py
  weights

=== aasist/config/ (if it exists) ===
  AASIST-L.conf
  AASIST.conf
  RawGATST_baseline.conf
  RawNet2_baseline.conf

=== .gitignore contents ===
# etc
_*
.*
exp_result/
config/
models/
depreciated/
LA/

# Byte-compiled / optimized / DLL files
__pycache__/
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
pip-wheel-metadata/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#  Usually these files are written by a python script from a template
#  before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log

In [9]:
import json
from pathlib import Path

# Copied verbatim from the repo's own RawNet2_baseline.conf,
# with only the three path fields swapped for Kaggle locations.
# Every other value (filts, first_conv, etc.) is exactly as shipped.

rawnet2_config = {
    "database_path"              : str(DATA_DIR) + "/",
    "asv_score_path"             : "ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt",
    "model_path"                 : str(MODEL_DIR) + "/",
    "batch_size"                 : 24,     # 32 in repo; 24 fits Kaggle T4 VRAM safely
    "lr"                         : 0.0001,
    "weight_decay"               : 0.0001,
    "num_epochs"                 : 100,
    "loss"                       : "CCE",
    "track"                      : "LA",
    "eval_output"                : str(EVAL_OUTPUT),
    "cudnn_deterministic_toggle" : "True",
    "cudnn_benchmark_toggle"     : "False",

    "model_config": {
        "architecture" : "RawNet2Spoof",  # matches models/RawNet2Spoof.py
        "nb_samp"      : 64600,
        "first_conv"   : 1024,            # repo's value (not 128 — that's the eurecom version)
        "in_channels"  : 1,
        "filts"        : [20, [20, 20], [20, 128], [128, 128]],
        "blocks"       : [2, 4],
        "nb_fc_node"   : 1024,
        "gru_node"     : 1024,
        "nb_gru_layer" : 3,
        "nb_classes"   : 2
    },

    "optim_config": {
        "optimizer"    : "adam",
        "amsgrad"      : "False",
        "base_lr"      : 0.0001,
        "lr_min"       : 0.000005,
        "betas"        : [0.9, 0.999],
        "weight_decay" : 0.0001,
        "scheduler"    : "cosine"
    }
}

with open(CONFIG_PATH, "w") as f:
    json.dump(rawnet2_config, f, indent=2)

# Quick sanity checks
with open(CONFIG_PATH) as f:
    check = json.load(f)

assert check["model_config"]["architecture"] == "RawNet2Spoof"
assert (AASIST_DIR / "models" / "RawNet2Spoof.py").exists(), \
    "models/RawNet2Spoof.py missing — check AASIST_DIR"
assert Path(check["database_path"]).exists(), \
    f"database_path not found: {check['database_path']}"

print("✓ Config written")
print(f"  architecture : {check['model_config']['architecture']}")
print(f"  first_conv   : {check['model_config']['first_conv']}")
print(f"  filts        : {check['model_config']['filts']}")
print(f"  database_path: {check['database_path']}")
print(f"  model_path   : {check['model_path']}")
print("\n✓ All assertions passed — safe to run Cell 7 (training)")

✓ Config written
  architecture : RawNet2Spoof
  first_conv   : 1024
  filts        : [20, [20, 20], [20, 128], [128, 128]]
  database_path: /input/asvspoof2019_download/LA/LA/
  model_path   : /kaggle/working/project4/rawnet2_models/

✓ All assertions passed — safe to run Cell 7 (training)


In [10]:
from pathlib import Path

# np.float / np.int / np.complex / np.bool / np.object / np.str
# were all removed in NumPy 1.24. The AASIST repo pre-dates that.
# Fix every .py file in the repo in one pass.

replacements = [
    ("np.float)",   "float)"),
    ("np.float,",   "float,"),
    ("np.float ",   "float "),
    ("np.float\n",  "float\n"),
    ("np.int)",     "np.int64)"),
    ("np.int,",     "np.int64,"),
    ("np.int ",     "np.int64 "),
    ("np.complex)", "np.complex128)"),
    ("np.bool)",    "bool)"),
    ("np.bool,",    "bool,"),
    ("np.object)",  "object)"),
    ("np.str)",     "str)"),
]

py_files = list(AASIST_DIR.rglob("*.py"))
patched  = []

for fpath in py_files:
    original = fpath.read_text(encoding="utf-8")
    updated  = original
    for old, new in replacements:
        updated = updated.replace(old, new)
    if updated != original:
        fpath.write_text(updated, encoding="utf-8")
        patched.append(fpath.name)

if patched:
    print(f"✓ Patched {len(patched)} file(s): {patched}")
else:
    print("✓ No deprecated aliases found — already clean")

# Verify the specific line that crashed is fixed
eval_src = (AASIST_DIR / "evaluation.py").read_text()
assert "np.float)" not in eval_src, "np.float still present in evaluation.py!"
print("✓ evaluation.py confirmed clean")
print("\n→ Run Cell 7 to start training")

✓ Patched 1 file(s): ['evaluation.py']
✓ evaluation.py confirmed clean

→ Run Cell 7 to start training


In [11]:
sys.path.insert(0, str(AASIST_DIR))

import random, numpy as np

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"✓ Seed fixed: {SEED}")

✓ Seed fixed: 1234


In [12]:
import subprocess, datetime

print("=" * 50)
print(" Training RawNet2...")
print(f" Started: {datetime.datetime.now()}")
print("=" * 50)

result = subprocess.run(
    [sys.executable, "main.py", "--config", str(CONFIG_PATH)],
    cwd=AASIST_DIR,
    capture_output=False,   # streams output to cell in real time
    text=True
)

print("=" * 50)
print(f" Finished: {datetime.datetime.now()}")
print(f" Return code: {result.returncode}")
print("=" * 50)

 Training RawNet2...
 Started: 2026-04-04 14:25:15.526438


2026-04-04 14:25:20.806457: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775312721.014227      88 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775312721.074624      88 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775312721.574399      88 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775312721.574448      88 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775312721.574452      88 computation_placer.cc:177] computation placer alr


CM SYSTEM
	EER		= 5.395222020 % (Equal error rate for countermeasure)

TANDEM
	min-tDCF		= 0.127557472

BREAKDOWN CM SYSTEM
	EER A07		= 1.239302405 % (Equal error rate for A07
	EER A08		= 2.437860500 % (Equal error rate for A08
	EER A09		= 0.179950347 % (Equal error rate for A09
	EER A10		= 1.524512575 % (Equal error rate for A10
	EER A11		= 0.814886199 % (Equal error rate for A11
	EER A12		= 1.996471187 % (Equal error rate for A12
	EER A13		= 0.302183276 % (Equal error rate for A13
	EER A14		= 0.587393446 % (Equal error rate for A14
	EER A15		= 1.157813785 % (Equal error rate for A15
	EER A16		= 1.076325165 % (Equal error rate for A16
	EER A17		= 12.209696792 % (Equal error rate for A17
	EER A18		= 18.460549321 % (Equal error rate for A18
	EER A19		= 2.403914287 % (Equal error rate for A19

CM SYSTEM
	EER		= 4.907855687 % (Equal error rate for countermeasure)

TANDEM
	min-tDCF		= 0.143034267

BREAKDOWN CM SYSTEM
	EER A07		= 3.463266346 % (Equal error rate for A07
	EER A08		= 3.456468

In [13]:
import re

if EVAL_OUTPUT.exists():
    lines = EVAL_OUTPUT.read_text().splitlines()
    eer_lines = [l for l in lines if "EER" in l]
    for l in eer_lines[-5:]:          # show last 5 EER entries
        print(l)
else:
    print("Eval output not found — training may not have completed.")

print("\n>> Target: ~5.1% EER (published RawNet2 result on ASVspoof 2019 LA)")


>> Target: ~5.1% EER (published RawNet2 result on ASVspoof 2019 LA)


In [14]:
if LOG_FILE.exists():
    log_text = LOG_FILE.read_text()
    matches = re.findall(r"best.*?EER.*?[\d.]+%?", log_text, re.IGNORECASE)
    for m in matches[-10:]:
        print(m)
else:
    print("Log file not found.")

Log file not found.


In [15]:
import shutil

OUTPUT_DIR = Path("/kaggle/working")

# Copy eval output and log so they persist after session ends
for src in [EVAL_OUTPUT, LOG_FILE]:
    if src.exists():
        shutil.copy(src, OUTPUT_DIR / src.name)
        print(f"✓ Saved {src.name}")

# Optionally copy best model checkpoint (can be large)
best_ckpts = list(MODEL_DIR.glob("*.pth"))
if best_ckpts:
    best = max(best_ckpts, key=lambda p: p.stat().st_mtime)
    shutil.copy(best, OUTPUT_DIR / best.name)
    print(f"✓ Saved checkpoint: {best.name}")
else:
    print("No checkpoints found yet.")

✓ Saved rawnet2_eval_output.txt
No checkpoints found yet.
